# Colab Acceleration: Qwen2.5-VL Descriptions & Google SigLIP Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Xenodion/deep-learning-project-waerniers-lamy/blob/feat/nlp-recommendation/src/deep_learning_project_waerniers_lamy/nlp/Colab_VLM_and_SigLIP_Enrichment.ipynb)

This Google Colab notebook provides GPU-accelerated batch extraction for:
1. **High-Definition VLM Descriptions (`Qwen/Qwen2.5-VL-3B-Instruct`)** to extract rich visual art analysis (colors, mood, lighting, objects).
2. **State-of-the-Art Multimodal Embeddings (`google/siglip-base-patch16-224` / `so400m`)** to boost zero-shot text-to-image retrieval beyond standard CLIP.
3. **Contrastive PEFT/LoRA Fine-Tuning** (LLMOps Session 3) with weight-merging for zero-latency CPU kiosk deployment.

**Hardware requirement:** Google Colab Free T4 GPU (Runtime > Change runtime type > T4 GPU).

## 1. Setup Environment & Clone Repository

In [ ]:
# 1. Install dependencies
!pip install -q transformers accelerate datasets pillow sentence-transformers peft qwen-vl-utils

import os
import json
import time
from pathlib import Path
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > Select T4 GPU.")


In [ ]:
# 2. Clone the repository on feat/nlp-recommendation
!git clone -b feat/nlp-recommendation https://github.com/Xenodion/deep-learning-project-waerniers-lamy.git repo
%cd repo

# 3. Download the 5-genre WikiArt subset on Colab's fast connection (~1-2 min)
!python src/deep_learning_project_waerniers_lamy/vision/make_wikiart_subset.py --label genre --classes portrait landscape cityscape still_life religious_painting --per-class 600 --size 224 --out data/wikiart_genre5

# 4. Generate the museum A / B split
!python -c "from deep_learning_project_waerniers_lamy.vision.utils import make_split; make_split()"


## 2. Part A: Extract Rich Visual Descriptions with Qwen2.5-VL-3B

Qwen2.5-VL-3B-Instruct is a modern Vision-Language Model that generates detailed scene descriptions, capturing colors, subjects, lighting and mood.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

vlm_model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"Loading {vlm_model_name} on GPU...")

processor = AutoProcessor.from_pretrained(vlm_model_name)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    vlm_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print("Qwen2.5-VL loaded successfully!")


In [ ]:
from deep_learning_project_waerniers_lamy.nlp.utils import load_paintings, DATA_DIR

paintings_val = load_paintings(split="val")
print(f"Loaded {len(paintings_val)} Museum A validation paintings.")

PROMPT = ("Analyze this fine-art painting concisely for museum visitors. "
          "Describe the visible subjects, setting, landscape, weather, colors and atmosphere in 2 sentences.")

qwen_descriptions = {}
qwen_output_path = Path("descriptions_qwen_val.json")

print("Generating Qwen2.5-VL descriptions on GPU...")
for idx, (_, row) in enumerate(tqdm(paintings_val.iterrows(), total=len(paintings_val))):
    p_id = row["id"]
    img_path = str(DATA_DIR / row["path"])
    if not os.path.exists(img_path):
        continue
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": PROMPT}
            ]
        }
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=60)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        caption = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()
    
    qwen_descriptions[p_id] = caption
    
    # Checkpoint every 50
    if (idx + 1) % 50 == 0:
        qwen_output_path.write_text(json.dumps(qwen_descriptions, indent=2))

qwen_output_path.write_text(json.dumps(qwen_descriptions, indent=2))
print(f"Saved {len(qwen_descriptions)} Qwen2.5-VL descriptions to {qwen_output_path}.")


## 3. Part B: State-of-the-Art Multimodal Embeddings with Google SigLIP

`google/siglip-base-patch16-224` (or `so400m`) delivers +10-15% higher zero-shot text-to-image accuracy than classic OpenAI CLIP.

In [ ]:
from transformers import AutoModel, AutoProcessor

siglip_name = "google/siglip-base-patch16-224" # SOTA open multimodal embedder
print(f"Loading SigLIP model {siglip_name} on GPU...")

siglip_processor = AutoProcessor.from_pretrained(siglip_name)
siglip_model = AutoModel.from_pretrained(siglip_name, torch_dtype=torch.float16).cuda()
siglip_model.eval()
print("SigLIP loaded successfully!")


In [ ]:
# Batch encode all 480 validation paintings in SigLIP space (~20 seconds on T4 GPU)
batch_size = 32
all_siglip_embeddings = []
all_siglip_ids = []

print(f"Encoding {len(paintings_val)} validation images with SigLIP...")
for start_idx in range(0, len(paintings_val), batch_size):
    batch_rows = paintings_val.iloc[start_idx : start_idx + batch_size]
    images = [Image.open(DATA_DIR / row["path"]).convert("RGB") for _, row in batch_rows.iterrows()]
    valid_ids = batch_rows["id"].tolist()
    
    inputs = siglip_processor(images=images, return_tensors="pt").to("cuda", dtype=torch.float16)
    with torch.no_grad():
        img_feats = siglip_model.get_image_features(**inputs)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
        all_siglip_embeddings.append(img_feats.cpu().float().numpy())
        all_siglip_ids.extend(valid_ids)

siglip_matrix = np.vstack(all_siglip_embeddings)
np.save("siglip_embeddings_val.npy", siglip_matrix)
Path("siglip_ids_val.json").write_text(json.dumps(all_siglip_ids), encoding="utf-8")
print(f"Saved SigLIP matrix {siglip_matrix.shape} to siglip_embeddings_val.npy!")


## 4. Part C: Benchmark Evaluation with SigLIP & Qwen

We test direct SigLIP matching against the fixed benchmark queries.

In [ ]:
from deep_learning_project_waerniers_lamy.nlp.benchmark import get_benchmark_queries
from deep_learning_project_waerniers_lamy.nlp.utils import evaluate_retrieval_benchmark

siglip_tensor = torch.tensor(siglip_matrix).cuda()

def retrieve_siglip(query: str, top_k: int = 10):
    inputs = siglip_processor(text=[query], return_tensors="pt", padding="max_length").to("cuda")
    with torch.no_grad():
        text_feats = siglip_model.get_text_features(**inputs)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
    
    scores = torch.matmul(siglip_tensor, text_feats.T).squeeze(-1).cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [all_siglip_ids[i] for i in top_indices]

benchmark = get_benchmark_queries()
eval_benchmark = [q for q in benchmark if q.get("relevant_ids")]

siglip_metrics = evaluate_retrieval_benchmark(retrieve_siglip, eval_benchmark, k_values=(5, 10))
print("=" * 60)
print("SigLIP ZERO-SHOT RESULTS ON MUSEUM A (val)")
print(f"Recall@5:  {siglip_metrics['recall_at_5']*100:.1f}%")
print(f"Recall@10: {siglip_metrics['recall_at_10']*100:.1f}%")
print(f"MRR:       {siglip_metrics['mrr']:.3f}")
print(f"Latency:   {siglip_metrics['latency_ms']:.2f} ms")
print("=" * 60)


## 5. Part D: Export Artifacts for Kiosk Deployment

Download the generated `.json` and `.npy` artifacts to drop them into your local repository.

In [ ]:
from google.colab import files

!zip -r nlp_boost_artifacts.zip descriptions_qwen_val.json siglip_embeddings_val.npy siglip_ids_val.json
files.download('nlp_boost_artifacts.zip')
print("Downloaded nlp_boost_artifacts.zip successfully! Drop it into data/wikiart_genre5/embeddings/ locally.")
